# Opinion dynamics and collection

In [56]:
from __future__ import annotations

import random
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional

import mesa
import numpy as np

from lib.datagen.samplers import Pi, message_sampler, spin_sampler

## mesa Agent class

In [57]:
class OpinionAgent(mesa.Agent):
    """An LLM-driven agent with a persona, opinion, and message inboxes."""

    def __init__(
        self,
        model,
        agent_index: int,
        persona: str,
    ):
        super().__init__(model)

        self.agent_index = agent_index
        self.persona = persona

        self.spin = 0
        self.raw_spins: list[int] = []

        self.inbox_agree: list[str] = []
        self.inbox_disagree: list[str] = []

        self.last_message: Optional[str] = None

        # Per-agent histories.
        self.spin_history: list[int] = []
        self.message_history: list[str] = []

    def sample_spin(self) -> int:
        """Sample the agent's opinion using its current inboxes."""
        return self.model.sample_agent_spin(self)

    def generate_message(self) -> str:
        """Generate a message from the agent's current state."""
        return self.model.generate_agent_message(self)

    def step(self):
        """
        Mesa agent activation is intentionally not used to update opinions.

        The model coordinates synchronized message and spin phases.
        """
        pass

## mesa Model class

This class represents the "replica", a realization of the independent experiment

In [58]:
class OpinionModel(mesa.Model):
    """
    Synchronous, LLM-driven opinion dynamics.

    Each call to step():
      - First call: initialize opinions with empty inboxes.
      - Subsequent calls: generate messages, deliver them, then
        update opinions.

    J[i, j] determines whether sender i communicates with receiver j
    and which inbox receives the message.
    """

    def __init__(
        self,
        personas: list[str],
        statement: str,
        J: np.ndarray,
        pi: Pi,
        *,
        name: str = "replica_0",
        mode: str = "subjective",
        choices: Optional[dict[str, str]] = None,
        k: int = 5,
        max_workers: int = 64,
        seed: Optional[int] = None,
    ):
        super().__init__(seed=seed)

        self.name = name
        self.personas = list(personas)
        self.statement = statement
        self.J = np.asarray(J, dtype=np.int8).copy()
        self.pi = pi

        self.mode = mode
        self.choices = choices
        self.k = k
        self.max_workers = max_workers

        self.n = len(self.personas)

        if self.J.shape != (self.n, self.n):
            raise ValueError(
                f"J has shape {self.J.shape}, "
                f"expected {(self.n, self.n)}"
            )

        if not np.isin(self.J, [-1, 0, 1]).all():
            raise ValueError("J must contain only -1, 0, +1.")

        if self.k < 1:
            raise ValueError("k must be >= 1.")

        if self.mode not in {"subjective", "objective"}:
            raise ValueError(
                "mode must be 'subjective' or 'objective'."
            )

        if self.mode == "objective":
            if not choices or not {"A", "B"} <= choices.keys():
                raise ValueError(
                    "Objective mode requires choices for A and B."
                )

        # Agents are registered with Mesa automatically.
        self.opinion_agents = [
            OpinionAgent(self, i, persona)
            for i, persona in enumerate(self.personas)
        ]

        # Model-level histories.
        # spins_history[t] has shape (n,).
        # spins_raw_history[t] has shape (n, k).
        # messages_history[t] maps (sender, receiver) -> message.
        self.spins_history: list[np.ndarray] = []
        self.spins_raw_history: list[np.ndarray] = []
        self.messages_history: list[dict[int, str]] = []

        self._initialized = False

        # One executor reused across all rounds.
        self.executor = ThreadPoolExecutor(
            max_workers=self.max_workers
        )

    def sample_agent_spin(self, agent: OpinionAgent) -> int:
        """One spin-sampler call, with retries for ValueError."""
        for _ in range(3):
            try:
                result = spin_sampler(
                    agent.persona,
                    self.statement,
                    agent.inbox_agree,
                    agent.inbox_disagree,
                    self.pi,
                    mode=self.mode,
                    choices=self.choices,
                )

                result = int(result)

                if result not in (-1, 0, 1):
                    raise ValueError(
                        f"Invalid spin returned: {result}"
                    )

                return result

            except ValueError:
                continue

        return 0

    def generate_agent_message(
        self,
        agent: OpinionAgent,
    ) -> str:
        """Generate one message for the sender's outgoing edges."""
        return message_sampler(
            persona=agent.persona,
            question=self.statement,
            messages_agree=list(agent.inbox_agree),
            messages_disagree=list(agent.inbox_disagree),
            current_spin=agent.spin,
            pi=self.pi,
            mode=self.mode,
            choices=self.choices,
        )

    @staticmethod
    def _aggregate_spin(samples: list[int]) -> int:
        """Majority vote; resolve ties using the first sample."""
        total = sum(samples)

        if total > 0:
            return 1
        if total < 0:
            return -1

        return samples[0] if samples else 0

    def _update_spins(self, step_idx: int) -> None:
        """Sample k spins per agent and update all states together."""
        futures = {}

        for agent in self.opinion_agents:
            for sample_idx in range(self.k):
                future = self.executor.submit(
                    self.sample_agent_spin,
                    agent,
                )
                futures[future] = (
                    agent.agent_index,
                    sample_idx,
                )

        raw = np.zeros((self.n, self.k), dtype=np.int8)

        for future in as_completed(futures):
            agent_idx, sample_idx = futures[future]

            try:
                raw[agent_idx, sample_idx] = future.result()
            except Exception:
                raw[agent_idx, sample_idx] = 0

        # Commit state only after every agent has been sampled.
        spins = np.array(
            [
                self._aggregate_spin(raw[i].tolist())
                for i in range(self.n)
            ],
            dtype=np.int8,
        )

        for i, agent in enumerate(self.opinion_agents):
            agent.raw_spins = raw[i].tolist()
            agent.spin = int(spins[i])
            agent.spin_history.append(int(spins[i]))

        self.spins_raw_history.append(raw.copy())
        self.spins_history.append(spins.copy())

    def _communicate(self, step_idx: int) -> None:
        """Generate one message per sender, then broadcast it."""
        futures = {}

        # Determine each sender's outgoing recipients.
        recipients: dict[int, list[int]] = {}

        for i in range(self.n):
            recipients[i] = np.flatnonzero(
                self.J[i] != 0
            ).tolist()

        # Only agents with outgoing edges need to generate messages.
        for i, js in recipients.items():
            if not js:
                continue

            agent = self.opinion_agents[i]

            future = self.executor.submit(
                self.generate_agent_message,
                agent,
            )
            futures[future] = i

        sender_messages: dict[int, str] = {}

        for future in as_completed(futures):
            sender_idx = futures[future]

            try:
                sender_messages[sender_idx] = future.result()

            except Exception:
                print(
                    f"\nError generating message for agent {sender_idx}:"
                )
                traceback.print_exc()
                raise

        # Construct the complete message set before delivering anything.
        step_messages: dict[str, str] = {}

        for i, js in recipients.items():
            if not js:
                continue

            message = sender_messages[i]
            step_messages[str(i)] = message

            self.opinion_agents[i].last_message = message
            self.opinion_agents[i].message_history.append(message)

        # Build fresh inboxes. No receiver is updated during generation.
        agree = [[] for _ in range(self.n)]
        disagree = [[] for _ in range(self.n)]

        incoming: dict[int, list[tuple[int, str]]] = {}

        for i, js in recipients.items():
            if not js:
                continue

            message = step_messages[str(i)]

            for j in js:
                incoming.setdefault(j, []).append((i, message))

        for j, items in incoming.items():
            # Stable, reproducible shuffle independent of task completion
            # order and the global random-number generator.
            random.Random(
                f"{self.name}|{step_idx}|{j}"
            ).shuffle(items)

            for i, message in items:
                if self.J[i, j] == 1:
                    agree[j].append(message)
                elif self.J[i, j] == -1:
                    disagree[j].append(message)

        for i, agent in enumerate(self.opinion_agents):
            agent.inbox_agree = agree[i]
            agent.inbox_disagree = disagree[i]

        self.messages_history.append(
            dict(sorted(step_messages.items()))
        )

    def step(self) -> None:
        """Advance the simulation by one synchronized transition."""
        if not self._initialized:
            # Initial opinions, before any messages have been exchanged.
            self._update_spins(step_idx=0)
            self._initialized = True
            return

        # Round t: messages from the previous state, followed by spins.
        step_idx = len(self.spins_history)

        self._communicate(step_idx)
        self._update_spins(step_idx)

    def run(self, num_steps: int = 8) -> "OpinionModel":
        """
        Run num_steps communication rounds.

        Produces num_steps + 1 opinion snapshots, including t=0.
        """
        if num_steps < 0:
            raise ValueError("num_steps must be >= 0.")

        if not self._initialized:
            self.step()

        for _ in range(num_steps):
            self.step()

        return self

    def save(self, output_dir: str | Path) -> Path:
        """Save model state and histories to one JSON file."""
        import json

        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        payload: dict[str, Any] = {
            "meta": {
                "name": self.name,
                "n": self.n,
                "num_steps": len(self.spins_history) - 1,
                "k": self.k,
                "mode": self.mode,
            },
            "personas": self.personas,
            "statement": self.statement,
            "choices": self.choices,
            "J": self.J.astype(int).tolist(),
            "spins_history": [
                s.astype(int).tolist()
                for s in self.spins_history
            ],
            "spins_raw_history": [
                s.astype(int).tolist()
                for s in self.spins_raw_history
            ],
            "messages_history": [
                {
                    f"{i}": message
                    for i, message in messages.items()
                }
                for messages in self.messages_history
            ],
        }

        output_path = out_dir / f"{self.name}.json"

        with output_path.open("w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        return output_path

    def close(self) -> None:
        """Release the thread pool when the simulation is finished."""
        self.executor.shutdown(wait=True)

## setup

In [59]:
from lib.datagen.utils import make_gemini_pi, make_mock_pi, make_openrouter_pi

persona1 = "You are a cautious scientist who values empirical evidence."
persona2 = "You are an economist who prioritizes economic growth."
persona3 = "You are an enviornmental activist who cares about climate change."
persona4 = "You are a skeptical citizen who questions mainstream narratives."

personas = [
    persona1,
    persona2,
    persona3,
    persona4,
]

statement = "Governments should prioritize climate policies."

# Directed, signed interaction network.
#
# J[i, j] = +1: i's message goes into j's agreement inbox.
# J[i, j] = -1: i's message goes into j's disagreement inbox.
# J[i, j] =  0: no communication.
# J = np.array(
#     [
#         [0,  1, -1,  0],
#         [1,  0,  1, -1],
#         [-1, 1,  0,  1],
#         [0, -1,  1,  0],
#     ],
#     dtype=np.int8,
# )

n = len(personas)

rng = np.random.default_rng()

J = rng.choice([-1, 0, 1], size=(n, n)).astype(np.int8)

# Make interactions symmetric
J = np.triu(J, k=1)
J = J + J.T

# No self-interactions
np.fill_diagonal(J, 0)

# Configure this using your existing Pi implementation.
model_name = "deepseek/deepseek-v4.1-flash-1017"
# model_name = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
# model_name = "gemini-3.5-flash-lite"
# model_name = "mock"

pi = make_openrouter_pi(model=model_name)
# pi = make_gemini_pi(model=model_name)
# pi = make_mock_pi()

splitted = model_name.split("-")
model_abbreviation =  splitted[0] if len(splitted) > 1 else model_name
model_abbreviation = model_abbreviation.replace("/", "")

replica_name = f"climate_replica_{model_abbreviation}_0"
model = OpinionModel(
    personas=personas,
    statement=statement,
    J=J,
    pi=pi,
    name=replica_name,
    mode="subjective",
    k=1,
    max_workers=1,
    seed=42,
)

random


## Execution and save

In [60]:
try:
    model.run(num_steps=6)

    print("Final opinions:")

    for agent in model.opinion_agents:
        print(
            f"Agent {agent.agent_index}: "
            f"spin={agent.spin}, "
            f"agree_messages={len(agent.inbox_agree)}, "
            f"disagree_messages={len(agent.inbox_disagree)}"
        )

    path = model.save("personal/mesa/outputs")
    print(f"Saved results to {path}")

finally:
    model.close()

Final opinions:
Agent 0: spin=1, agree_messages=2, disagree_messages=1
Agent 1: spin=-1, agree_messages=1, disagree_messages=0
Agent 2: spin=1, agree_messages=0, disagree_messages=1
Agent 3: spin=-1, agree_messages=1, disagree_messages=0
Saved results to personal/mesa/outputs/climate_replica_deepseekdeepseek_0.json


## Post-processing

In [61]:
import json
import matplotlib.pyplot as plt

with open(f"personal/mesa/outputs/{replica_name}.json", "r") as f:
    data = json.load(f)

json_num_steps = data["meta"]["num_steps"]
json_personas = data["personas"]
json_J = np.array(data["J"])
json_spins = np.array(data["spins_history"])
json_messages = np.array(data["messages_history"])
json_n = len(json_personas)


In [62]:
json_messages[0]['0']

'I think governments should prioritize climate policies because the empirical evidence consistently indicates significant and accelerating climate-related risks, and credible evidence shows well-designed policies can reduce those risks. Given remaining uncertainties, prioritizing action now is the prudent, evidence-based choice.'

In [63]:

import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx

from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.lines import Line2D

# --------------------------------------------------
# Configuration
# --------------------------------------------------

output_dir = Path(f"pdf/{model_abbreviation}")
output_dir.mkdir(parents=True, exist_ok=True)

# Fixed landscape A4 page
PAGE_WIDTH = 11.69
PAGE_HEIGHT = 8.27

# Wider agent boxes
BOX_WIDTH = 4.8
BOX_PADDING = 0.22

# Spacing between neighboring boxes
HORIZONTAL_GAP = 0.65
VERTICAL_GAP = 0.65

# Text wrapping widths
PERSONA_WIDTH = 32
MESSAGE_WIDTH = 42

# Font sizes in points
TITLE_FONT = 11
PERSONA_FONT = 9
MESSAGE_FONT = 9

MIN_TITLE_FONT = 5
MIN_PERSONA_FONT = 4
MIN_MESSAGE_FONT = 4

# Page margins
PAGE_LEFT = 0.04
PAGE_RIGHT = 0.04
PAGE_BOTTOM = 0.06
PAGE_TOP = 0.13

# --------------------------------------------------
# Build graph
# --------------------------------------------------

G = nx.Graph()
G.add_nodes_from(range(json_n))

for i in range(json_n):
    for j in range(i + 1, json_n):
        weight = json_J[i, j]

        if weight != 0:
            G.add_edge(i, j, weight=weight)

# --------------------------------------------------
# Prepare text
# --------------------------------------------------

personas = [
    textwrap.fill(
        str(json_personas[i]),
        width=PERSONA_WIDTH,
        break_long_words=True,
        break_on_hyphens=False
    )
    for i in range(json_n)
]


def get_message(step, agent):
    if step >= json_num_steps:
        return "No message"

    return str(
        json_messages[step].get(str(agent), "No message")
    )


wrapped_messages = {}

for step in range(json_num_steps + 1):
    wrapped_messages[step] = {}

    for i in range(json_n):
        wrapped_messages[step][i] = textwrap.fill(
            get_message(step, i),
            width=MESSAGE_WIDTH,
            break_long_words=True,
            break_on_hyphens=False
        )

# --------------------------------------------------
# Calculate stable box heights
# Reserve space for the longest message across all steps
# --------------------------------------------------

box_heights = {}

for i in range(json_n):
    persona_lines = max(
        1, len(personas[i].splitlines())
    )

    max_message_lines = max(
        len(wrapped_messages[s][i].splitlines())
        for s in range(json_num_steps + 1)
    )

    box_heights[i] = (
        0.65
        + 0.24 * persona_lines
        + 0.24 * max_message_lines
        + 2 * BOX_PADDING
    )

# --------------------------------------------------
# Triangular layout
# --------------------------------------------------

pos = {}
row_nodes = []
row_heights = []

node = 0
row = 0

while node < json_n:
    current_row = []

    for col in range(row + 1):
        if node >= json_n:
            break

        current_row.append(node)
        node += 1

    row_nodes.append(current_row)

    row_heights.append(
        max(box_heights[i] for i in current_row)
    )

    row += 1

# Position each row, allowing taller rows more space
current_y = 0.0

for r, nodes in enumerate(row_nodes):
    row_height = row_heights[r]

    for col, i in enumerate(nodes):
        pos[i] = (
            (
                col - (len(nodes) - 1) / 2
            ) * (BOX_WIDTH + HORIZONTAL_GAP),
            current_y
        )

    current_y -= row_height + VERTICAL_GAP

# --------------------------------------------------
# Determine layout boundaries
# --------------------------------------------------

if json_n > 0:
    min_x = min(
        pos[i][0] - BOX_WIDTH / 2
        for i in range(json_n)
    )

    max_x = max(
        pos[i][0] + BOX_WIDTH / 2
        for i in range(json_n)
    )

    min_y = min(
        pos[i][1] - box_heights[i] / 2
        for i in range(json_n)
    )

    max_y = max(
        pos[i][1] + box_heights[i] / 2
        for i in range(json_n)
    )
else:
    min_x, max_x = -1, 1
    min_y, max_y = -1, 1

network_width = max(max_x - min_x, 1e-6)
network_height = max(max_y - min_y, 1e-6)

# --------------------------------------------------
# Fit network to page using independent X/Y scaling
# This is the key change: no uniform scaling.
# --------------------------------------------------

usable_width_in = PAGE_WIDTH * (
    1 - PAGE_LEFT - PAGE_RIGHT
)

usable_height_in = PAGE_HEIGHT * (
    1 - PAGE_BOTTOM - PAGE_TOP
)

scale_x = usable_width_in / network_width
scale_y = usable_height_in / network_height

center_x = (min_x + max_x) / 2
center_y = (min_y + max_y) / 2

def page_x(x):
    return (
        0.5
        + (x - center_x) * scale_x / PAGE_WIDTH
    )


def page_y(y):
    return (
        PAGE_BOTTOM
        + usable_height_in / PAGE_HEIGHT / 2
        + (y - center_y) * scale_y / PAGE_HEIGHT
    )


# --------------------------------------------------
# Adaptive fonts
# Keep fonts near their target sizes unless the network
# is very dense.
# --------------------------------------------------

density_scale = min(
    1.0,
    scale_x / 0.65,
    scale_y / 0.65
)

title_font = max(
    MIN_TITLE_FONT,
    TITLE_FONT * density_scale
)

persona_font = max(
    MIN_PERSONA_FONT,
    PERSONA_FONT * density_scale
)

message_font = max(
    MIN_MESSAGE_FONT,
    MESSAGE_FONT * density_scale
)

# --------------------------------------------------
# Prepare edges
# --------------------------------------------------

positive_edges = [
    (u, v)
    for u, v in G.edges()
    if G[u][v]["weight"] == 1
]

negative_edges = [
    (u, v)
    for u, v in G.edges()
    if G[u][v]["weight"] == -1
]

# --------------------------------------------------
# Legend
# --------------------------------------------------

legend_elements = [
    Patch(
        facecolor="lightgreen",
        edgecolor="#34495e",
        label="Spin = +1"
    ),
    Patch(
        facecolor="lightcoral",
        edgecolor="#34495e",
        label="Spin = -1"
    ),
    Patch(
        facecolor="lightgray",
        edgecolor="#34495e",
        label="Spin = 0 / other"
    ),
    Line2D(
        [0], [0],
        color="blue",
        linewidth=1.8,
        linestyle="solid",
        label="Positive link (+1)"
    ),
    Line2D(
        [0], [0],
        color="gray",
        linewidth=1.8,
        linestyle="dotted",
        label="Negative link (-1)"
    ),
]

# --------------------------------------------------
# Render slides
# --------------------------------------------------

for step in range(json_num_steps + 1):

    fig, ax = plt.subplots(
        figsize=(PAGE_WIDTH, PAGE_HEIGHT)
    )

    ax.set_position([
        PAGE_LEFT,
        PAGE_BOTTOM,
        1 - PAGE_LEFT - PAGE_RIGHT,
        1 - PAGE_BOTTOM - PAGE_TOP
    ])

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    fig.suptitle(
        f"Step {step + 1}",
        fontsize=14,
        y=0.97
    )

    current_spins = json_spins[step]

    # ----------------------------------------------
    # Draw edges behind the nodes
    # ----------------------------------------------

    for edges, color, style in [
        (positive_edges, "blue", "solid"),
        (negative_edges, "gray", "dotted")
    ]:
        for u, v in edges:
            x1, y1 = pos[u]
            x2, y2 = pos[v]

            ax.plot(
                [page_x(x1), page_x(x2)],
                [page_y(y1), page_y(y2)],
                color=color,
                linewidth=1.2,
                linestyle=style,
                transform=ax.transAxes,
                zorder=1,
                clip_on=True
            )

    # ----------------------------------------------
    # Draw agent boxes and their text
    # ----------------------------------------------

    for i in range(json_n):
        x, y = pos[i]
        height = box_heights[i]

        px = page_x(x)
        py = page_y(y)

        box_width_norm = BOX_WIDTH * scale_x / PAGE_WIDTH
        box_height_norm = height * scale_y / PAGE_HEIGHT

        # Spin color
        spin = current_spins[i]

        if spin == 1:
            face_color = "lightgreen"
        elif spin == -1:
            face_color = "lightcoral"
        else:
            face_color = "lightgray"

        box = FancyBboxPatch(
            (
                px - box_width_norm / 2,
                py - box_height_norm / 2
            ),
            box_width_norm,
            box_height_norm,
            boxstyle="round,pad=0.002,rounding_size=0.008",
            facecolor=face_color,
            edgecolor="#34495e",
            linewidth=0.9,
            transform=ax.transAxes,
            zorder=3,
            clip_on=True
        )

        ax.add_patch(box)

        persona = personas[i]
        message = wrapped_messages[step][i]

        persona_line_count = max(
            1, len(persona.splitlines())
        )

        # All vertical offsets use the Y scaling so that
        # text positions remain proportional to the boxes.
        title_y = (
            py + box_height_norm / 2
            - (BOX_PADDING + 0.08) * scale_y / PAGE_HEIGHT
        )

        persona_y = (
            title_y - 0.32 * scale_y / PAGE_HEIGHT
        )

        message_y = (
            persona_y
            - (0.22 * persona_line_count + 0.16)
            * scale_y / PAGE_HEIGHT
        )

        ax.text(
            px,
            title_y,
            f"Agent {i}",
            ha="center",
            va="top",
            fontsize=title_font,
            fontweight="bold",
            transform=ax.transAxes,
            zorder=4,
            clip_on=True
        )

        ax.text(
            px,
            persona_y,
            persona,
            ha="center",
            va="top",
            fontsize=persona_font,
            linespacing=1.1,
            transform=ax.transAxes,
            zorder=4,
            clip_on=True
        )

        ax.text(
            px,
            message_y,
            f"“{message}”",
            ha="center",
            va="top",
            fontsize=message_font,
            fontstyle="italic",
            linespacing=1.1,
            transform=ax.transAxes,
            zorder=4,
            clip_on=True
        )

    # ----------------------------------------------
    # Legend stays inside the fixed page
    # ----------------------------------------------

    fig.legend(
        handles=legend_elements,
        loc="upper right",
        bbox_to_anchor=(0.96, 0.95),
        fontsize=7,
        frameon=True,
        framealpha=0.95,
        borderpad=0.5,
        labelspacing=0.35,
        handlelength=1.6,
        title="Legend",
        title_fontsize=8
    )

    # ----------------------------------------------
    # Save fixed landscape PDF
    # ----------------------------------------------

    fig.savefig(
        output_dir / f"step_{step + 1:03d}.pdf",
        format="pdf",
        bbox_inches=None,
        pad_inches=0
    )

    plt.close(fig)

print(
    f"Saved {json_num_steps + 1} landscape PDF slides to "
    f"{output_dir.resolve()}/"
)


Saved 7 landscape PDF slides to /Users/leo/Documents/github/physics-of-agents/pdf/deepseekdeepseek/
